# Lesson 5.3 — Error Propagation Strategies Across Multi-Agent Systems

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Return structured error context (failure type, attempted query, partial results, alternatives)
- Distinguish access failures from valid empty results
- Have subagents recover locally and propagate only unresolved errors with partials
- Annotate synthesis output with coverage gaps for unavailable sources

**You will produce:** A structured subagent error object plus a coordinator recovery decision and coverage annotation.

## Concepts

**Structured error context** enables intelligent coordinator recovery. Sample Question 8: a timed-out web-search subagent should return the **failure type, attempted query, partial results, and potential alternatives** - not a generic "search unavailable".

Anti-patterns to reject:
- **Generic statuses** that hide context from the coordinator.
- **Silently suppressing errors** (returning empty results as success).
- **Terminating the entire workflow** on a single failure.

Patterns to apply:
- **Local recovery** inside subagents for transient failures; only propagate what can't be resolved locally, **with partial results and what was attempted**.
- **Access failure != valid empty result** - distinguish them so the coordinator decides correctly.
- **Coverage annotations** in synthesis output indicating which findings are well-supported vs which topic areas have gaps due to unavailable sources.

> **Reference:** [Anthropic - multi-agent reliability](https://docs.anthropic.com/en/docs/agents-and-tools/agent-sdk)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Build a structured subagent error

Everything the coordinator needs to choose a recovery path travels with the error.

In [ ]:
def subagent_error(failure_type, attempted, partial=None, alternatives=None):
    return {
        "ok": False,
        "failure_type": failure_type,          # timeout|unreachable|rate_limited
        "attempted_query": attempted,
        "partial_results": partial or [],
        "alternatives": alternatives or [],
    }

err = subagent_error("timeout", "AI in film production 2026",
                     partial=["1 source found"],
                     alternatives=["retry narrower query", "use cached corpus"])
import json; print(json.dumps(err, indent=2))

## Code-Along: Step 2 — Coordinator recovery decision

The coordinator proceeds with partials and records a gap rather than aborting.

In [ ]:
def coordinator_recover(err):
    if err["alternatives"]:
        plan = f"try alternative: {err['alternatives'][0]}"
    else:
        plan = "proceed with partial results"
    return {"action": plan, "keep_partials": err["partial_results"],
            "coverage_gap": err["attempted_query"]}

print(coordinator_recover(err))

## Code-Along: Step 3 — Access failure vs valid empty result

An empty but successful query is NOT an error; only access failures carry ok=False.

In [ ]:
def web_search(query, reachable=True, hits=0):
    if not reachable:
        return subagent_error("unreachable", query)
    return {"ok": True, "results": [f"hit {i}" for i in range(hits)]}

print("empty success:", web_search("obscure topic", hits=0))
print("access failure:", web_search("x", reachable=False)["failure_type"])

## Exercise

Add a `synthesize(findings, gaps)` function that emits a report with an explicit **Coverage** section listing well-supported areas vs gap areas (from unavailable sources). Assert that a single subagent timeout does NOT terminate synthesis and that the gap is annotated rather than hidden.

In [ ]:
# TODO: implement synthesize(findings, gaps) with a Coverage section
# TODO: assert synthesis still runs when one subagent errored
# TODO: assert the coverage gap appears in the output

## Key Takeaways

- Return failure type, attempted query, partial results, and alternatives - not generic statuses.
- Distinguish access failures from valid empty results so the coordinator decides correctly.
- Recover transient failures locally; propagate only unresolved errors with partials.
- Never suppress errors or abort the whole workflow on one failure - annotate coverage gaps.

---

**Next:** [Lesson 5.4 — Context in Large Codebase Exploration](./lesson-5.4-large-codebase-context.ipynb)